# Combined Luenberger Observer and State Feedback Control

This notebook provides an interactive tool to design and analyze a **combined Luenberger observer and state feedback controller** for a 2D LTI system.

## System Description

We consider a continuous-time LTI system:

$$
\dot{x}(t) = A x(t) + B u(t), \qquad y(t) = C x(t),
$$

where $A \in \mathbb{R}^{2\times 2}$, $B \in \mathbb{R}^{2\times 1}$, and $C \in \mathbb{R}^{1\times 2}$.

## Control Architecture

The system uses state feedback with an observer:

- **Observer**: $\dot{\hat{x}}(t) = A \hat{x}(t) + B u(t) + L (y(t) - \hat{y}(t))$, where $\hat{y} = C \hat{x}$
- **Controller**: $u(t) = -K \hat{x}(t) + \bar{N} r(t)$

The observer poles are determined by eigenvalues of $A - LC$. The controller poles are determined by eigenvalues of $A - BK$.

**Tuning method**: Click on the pole maps to place observer and controller poles interactively, or use sliders for manual adjustment.

In [ ]:
# Install required packages
!pip install numpy matplotlib scipy ipywidgets control

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import place_poles
import ipywidgets as widgets
from IPython.display import display, Markdown

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True

# Enable widget backend for interactive plots
try:
    get_ipython().run_line_magic('matplotlib', 'widget')
    _backend_ok = True
except Exception:
    _backend_ok = False

## Step 1: System Definition

Enter your system matrices using comma-separated values in row-major order:

In [ ]:
def parse_matrix(text, shape):
    """Parse comma-separated matrix values."""
    vals = [v.strip() for v in text.split(',') if v.strip() != '']
    expected = shape[0] * shape[1]
    if len(vals) != expected:
        raise ValueError(f'Expected {expected} values, got {len(vals)}')
    arr = np.array([float(v) for v in vals], dtype=float).reshape(shape)
    return arr

# System matrix inputs
A_text = widgets.Text(
    value='-1.1, 0.4, 0.25, 0',
    description='A:',
    layout=widgets.Layout(width='450px'),
    tooltip='A matrix (2x2): a11, a12, a21, a22'
)
B_text = widgets.Text(
    value='1, 0',
    description='B:',
    layout=widgets.Layout(width='450px'),
    tooltip='B matrix (2x1): b1, b2'
)
C_text = widgets.Text(
    value='-1, 0.8',
    description='C:',
    layout=widgets.Layout(width='450px'),
    tooltip='C matrix (1x2): c1, c2'
)

out_model = widgets.Output()

def update_model(_=None):
    out_model.clear_output()
    with out_model:
        try:
            A = parse_matrix(A_text.value, (2, 2))
            B = parse_matrix(B_text.value, (2, 1))
            C = parse_matrix(C_text.value, (1, 2))

            print('System Matrices:')
            print('A =')
            print(A)
            print('\nB =')
            print(B)
            print('\nC =')
            print(C)

            eigA = np.linalg.eigvals(A)
            print(f'\nOpen-loop poles (eig(A)): {eigA}')

            # Store globally
            globals()['A_current'] = A
            globals()['B_current'] = B
            globals()['C_current'] = C
            globals()['L_current'] = None
            globals()['K_current'] = None

        except Exception as e:
            print(f'Error: {e}')

A_text.observe(update_model, names='value')
B_text.observe(update_model, names='value')
C_text.observe(update_model, names='value')

display(widgets.VBox([
    widgets.HTML('<b>System Matrices (row-major order)</b>'),
    widgets.HTML('A (2×2): a<sub>11</sub>, a<sub>12</sub>, a<sub>21</sub>, a<sub>22</sub>'),
    A_text,
    widgets.HTML('B (2×1): b<sub>1</sub>, b<sub>2</sub>'),
    B_text,
    widgets.HTML('C (1×2): c<sub>1</sub>, c<sub>2</sub>'),
    C_text,
    out_model
]))

update_model()

## Step 2: Observer Pole Placement

Design observer gains by clicking on the pole map to place observer poles. This determines the observer convergence rate and damping.

In [ ]:
# Observer pole placement interface
l1_slider = widgets.FloatSlider(value=-1.5, min=-20.0, max=5.0, step=0.1, description='Real part:', continuous_update=False)
l2_slider = widgets.FloatSlider(value=1.5, min=0.0, max=10.0, step=0.1, description='Imag part:', continuous_update=False)

l1_text = widgets.FloatText(value=-1.5, description='Real exact:')
l2_text = widgets.FloatText(value=1.5, description='Imag exact:')

info_obs = widgets.HTML('Click in pole map to place observer poles.')
out_obs_design = widgets.Output()

_obs_updating = {'busy': False}

def _ensure_slider_range(slider, value):
    """Auto-expand slider bounds if value goes outside."""
    if value < slider.min:
        slider.min = min(value * 1.2, value - 1.0)
    if value > slider.max:
        slider.max = max(value * 1.2, value + 1.0)

def _ordered_branches(eigs_seq):
    """Track eigenvalue branches to avoid crossing lines."""
    if len(eigs_seq) == 0:
        return np.array([]), np.array([])
    b1 = [eigs_seq[0][0]]
    b2 = [eigs_seq[0][1]]
    for i in range(1, len(eigs_seq)):
        c0, c1 = eigs_seq[i][0], eigs_seq[i][1]
        p0, p1 = b1[-1], b2[-1]
        d_keep = abs(c0 - p0) + abs(c1 - p1)
        d_swap = abs(c1 - p0) + abs(c0 - p1)
        if d_keep <= d_swap:
            b1.append(c0)
            b2.append(c1)
        else:
            b1.append(c1)
            b2.append(c0)
    return np.array(b1), np.array(b2)

def _compute_observer_gain(A, C, real_part, imag_part):
    """Compute observer gain via pole placement."""
    desired_poles = np.array([real_part + imag_part * 1j, real_part - imag_part * 1j])
    try:
        L = place_poles(A.T, C.T, desired_poles).gain_matrix.T
        return L
    except:
        return None

def _set_l_values(l1, l2):
    """Update observer gain sliders."""
    _obs_updating['busy'] = True
    _ensure_slider_range(l1_slider, l1)
    _ensure_slider_range(l2_slider, l2)
    l1_slider.value = float(l1)
    l2_slider.value = float(l2)
    l1_text.value = float(l1)
    l2_text.value = float(l2)
    _obs_updating['busy'] = False

def _draw_observer_design(l1, l2):
    """Visualize observer pole placement."""
    A = globals().get('A_current')
    C = globals().get('C_current')
    
    if A is None or C is None:
        print('Please define system matrices first.')
        return

    # Compute eigenvalue branches for visualization
    npts = 400
    l1_vals = np.linspace(l1_slider.min, l1_slider.max, npts)
    eigs_l1 = []
    for ll1 in l1_vals:
        L_test = _compute_observer_gain(A, C, ll1, l2)
        if L_test is not None:
            eigs_l1.append(np.linalg.eigvals(A - L_test @ C))
    
    l2_vals = np.linspace(max(0.01, l2_slider.min), l2_slider.max, npts)
    eigs_l2 = []
    for ll2 in l2_vals:
        L_test = _compute_observer_gain(A, C, l1, ll2)
        if L_test is not None:
            eigs_l2.append(np.linalg.eigvals(A - L_test @ C))

    b1_l1, b2_l1 = _ordered_branches(eigs_l1) if len(eigs_l1) > 0 else (np.array([]), np.array([]))
    b1_l2, b2_l2 = _ordered_branches(eigs_l2) if len(eigs_l2) > 0 else (np.array([]), np.array([]))

    L_now = _compute_observer_gain(A, C, l1, l2)
    if L_now is None:
        print('Error: Could not compute observer gain.')
        return

    poles_now = np.linalg.eigvals(A - L_now @ C)
    globals()['L_current'] = L_now

    fig, ax = plt.subplots(1, 1, figsize=(9, 7))

    if len(b1_l1) > 0:
        ax.plot(b1_l1.real, b1_l1.imag, 'b-', linewidth=1.5, alpha=0.8, label='Real part sweep')
        ax.plot(b2_l1.real, b2_l1.imag, 'b-', linewidth=1.5, alpha=0.8)
    if len(b1_l2) > 0:
        ax.plot(b1_l2.real, b1_l2.imag, 'g--', linewidth=1.5, alpha=0.8, label='Imag part sweep')
        ax.plot(b2_l2.real, b2_l2.imag, 'g--', linewidth=1.5, alpha=0.8)
    
    ax.scatter(poles_now.real, poles_now.imag, s=150, marker='o', color='red', 
               edgecolors='darkred', linewidth=2, label='Observer poles', zorder=5)
    ax.axhline(0.0, linewidth=1, color='gray', alpha=0.5)
    ax.axvline(0.0, linewidth=1, color='gray', alpha=0.5)
    ax.set_title('Observer Pole Placement', fontsize=13, fontweight='bold')
    ax.set_xlabel('Real Part', fontsize=11)
    ax.set_ylabel('Imaginary Part', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)

    def on_click_obs(event):
        if event.inaxes != ax or event.xdata is None or event.ydata is None:
            return
        _set_l_values(float(event.xdata), float(abs(event.ydata)))
        info_obs.value = f'<b>Observer poles placed at:</b> σ = {event.xdata:.3f}, ω = {abs(event.ydata):.3f}'
        _update_obs_design()

    fig.canvas.mpl_connect('button_press_event', on_click_obs)
    plt.show()

    print(f'Observer Gain L:')
    print(L_now)
    print(f'\nObserver Poles (eig(A - LC)): {poles_now}')

def _update_obs_design(_=None):
    if _obs_updating['busy']:
        return
    out_obs_design.clear_output(wait=True)
    with out_obs_design:
        try:
            _draw_observer_design(l1_slider.value, l2_slider.value)
        except Exception as e:
            print(f'Error: {e}')

def _on_obs_slider_change(_=None):
    if _obs_updating['busy']:
        return
    _obs_updating['busy'] = True
    l1_text.value = float(l1_slider.value)
    l2_text.value = float(l2_slider.value)
    _obs_updating['busy'] = False
    _update_obs_design()

def _on_obs_text_change(_=None):
    if _obs_updating['busy']:
        return
    _set_l_values(l1_text.value, l2_text.value)
    _update_obs_design()

l1_slider.observe(_on_obs_slider_change, names='value')
l2_slider.observe(_on_obs_slider_change, names='value')
l1_text.observe(_on_obs_text_change, names='value')
l2_text.observe(_on_obs_text_change, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Observer Pole Placement</b>'),
    l1_slider,
    l2_slider,
    widgets.HTML('Or enter exact values:'),
    widgets.HBox([l1_text, l2_text]),
    info_obs,
    out_obs_design
]))

_update_obs_design()

## Step 3: Controller Pole Placement

Design the state feedback controller $u = -Kx$ by placing closed-loop poles via $A - BK$.

In [ ]:
# Controller pole placement interface
k1_slider = widgets.FloatSlider(value=-0.5, min=-20.0, max=10.0, step=0.1, description='Real part:', continuous_update=False)
k2_slider = widgets.FloatSlider(value=2.0, min=0.0, max=10.0, step=0.1, description='Imag part:', continuous_update=False)

k1_text = widgets.FloatText(value=-0.5, description='Real exact:')
k2_text = widgets.FloatText(value=2.0, description='Imag exact:')

info_ctrl = widgets.HTML('Click in pole map to place controller poles.')
out_ctrl_design = widgets.Output()

_ctrl_updating = {'busy': False}

def _compute_controller_gain(A, B, real_part, imag_part):
    """Compute controller gain via pole placement."""
    desired_poles = np.array([real_part + imag_part * 1j, real_part - imag_part * 1j])
    try:
        K = place_poles(A, B, desired_poles).gain_matrix
        return K
    except:
        return None

def _set_k_values(k1, k2):
    """Update controller gain sliders."""
    _ctrl_updating['busy'] = True
    _ensure_slider_range(k1_slider, k1)
    _ensure_slider_range(k2_slider, k2)
    k1_slider.value = float(k1)
    k2_slider.value = float(k2)
    k1_text.value = float(k1)
    k2_text.value = float(k2)
    _ctrl_updating['busy'] = False

def _draw_controller_design(k1, k2):
    """Visualize controller pole placement."""
    A = globals().get('A_current')
    B = globals().get('B_current')
    
    if A is None or B is None:
        print('Please define system matrices first.')
        return

    # Compute eigenvalue branches
    npts = 400
    k1_vals = np.linspace(k1_slider.min, k1_slider.max, npts)
    eigs_k1 = []
    for kk1 in k1_vals:
        K_test = _compute_controller_gain(A, B, kk1, k2)
        if K_test is not None:
            eigs_k1.append(np.linalg.eigvals(A - B @ K_test))
    
    k2_vals = np.linspace(max(0.01, k2_slider.min), k2_slider.max, npts)
    eigs_k2 = []
    for kk2 in k2_vals:
        K_test = _compute_controller_gain(A, B, k1, kk2)
        if K_test is not None:
            eigs_k2.append(np.linalg.eigvals(A - B @ K_test))

    b1_k1, b2_k1 = _ordered_branches(eigs_k1) if len(eigs_k1) > 0 else (np.array([]), np.array([]))
    b1_k2, b2_k2 = _ordered_branches(eigs_k2) if len(eigs_k2) > 0 else (np.array([]), np.array([]))

    K_now = _compute_controller_gain(A, B, k1, k2)
    if K_now is None:
        print('Error: Could not compute controller gain.')
        return

    poles_now = np.linalg.eigvals(A - B @ K_now)
    globals()['K_current'] = K_now

    fig, ax = plt.subplots(1, 1, figsize=(9, 7))

    if len(b1_k1) > 0:
        ax.plot(b1_k1.real, b1_k1.imag, 'b-', linewidth=1.5, alpha=0.8, label='Real part sweep')
        ax.plot(b2_k1.real, b2_k1.imag, 'b-', linewidth=1.5, alpha=0.8)
    if len(b1_k2) > 0:
        ax.plot(b1_k2.real, b1_k2.imag, 'g--', linewidth=1.5, alpha=0.8, label='Imag part sweep')
        ax.plot(b2_k2.real, b2_k2.imag, 'g--', linewidth=1.5, alpha=0.8)
    
    ax.scatter(poles_now.real, poles_now.imag, s=150, marker='s', color='blue', 
               edgecolors='darkblue', linewidth=2, label='Controller poles', zorder=5)
    ax.axhline(0.0, linewidth=1, color='gray', alpha=0.5)
    ax.axvline(0.0, linewidth=1, color='gray', alpha=0.5)
    ax.set_title('Controller Pole Placement', fontsize=13, fontweight='bold')
    ax.set_xlabel('Real Part', fontsize=11)
    ax.set_ylabel('Imaginary Part', fontsize=11)
    ax.legend(loc='best', fontsize=10)
    ax.grid(True, alpha=0.3)

    def on_click_ctrl(event):
        if event.inaxes != ax or event.xdata is None or event.ydata is None:
            return
        _set_k_values(float(event.xdata), float(abs(event.ydata)))
        info_ctrl.value = f'<b>Controller poles placed at:</b> σ = {event.xdata:.3f}, ω = {abs(event.ydata):.3f}'
        _update_ctrl_design()

    fig.canvas.mpl_connect('button_press_event', on_click_ctrl)
    plt.show()

    print(f'Controller Gain K:')
    print(K_now)
    print(f'\nController Poles (eig(A - BK)): {poles_now}')

def _update_ctrl_design(_=None):
    if _ctrl_updating['busy']:
        return
    out_ctrl_design.clear_output(wait=True)
    with out_ctrl_design:
        try:
            _draw_controller_design(k1_slider.value, k2_slider.value)
        except Exception as e:
            print(f'Error: {e}')

def _on_ctrl_slider_change(_=None):
    if _ctrl_updating['busy']:
        return
    _ctrl_updating['busy'] = True
    k1_text.value = float(k1_slider.value)
    k2_text.value = float(k2_slider.value)
    _ctrl_updating['busy'] = False
    _update_ctrl_design()

def _on_ctrl_text_change(_=None):
    if _ctrl_updating['busy']:
        return
    _set_k_values(k1_text.value, k2_text.value)
    _update_ctrl_design()

k1_slider.observe(_on_ctrl_slider_change, names='value')
k2_slider.observe(_on_ctrl_slider_change, names='value')
k1_text.observe(_on_ctrl_text_change, names='value')
k2_text.observe(_on_ctrl_text_change, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Controller Pole Placement</b>'),
    k1_slider,
    k2_slider,
    widgets.HTML('Or enter exact values:'),
    widgets.HBox([k1_text, k2_text]),
    info_ctrl,
    out_ctrl_design
]))

_update_ctrl_design()

## Step 4: Closed-Loop Simulation

Simulate the combined observer and controller with optional reference tracking. The plots show:
- **Left**: Observer convergence (true state vs estimated state)
- **Right**: Closed-loop system response to reference input

In [ ]:
# Reference tracking controls
use_feedforward = widgets.Checkbox(value=False, description='Use Feedforward Term')
nbar_slider = widgets.FloatSlider(value=0.0, min=-2.0, max=2.0, step=0.05, 
                                   description='Feedforward Nbar:', continuous_update=False)
ref_magnitude = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1,
                                    description='Reference Value:', continuous_update=False)

out_simulation = widgets.Output()

def _compute_nbar_optimal(A, B, K, C):
    """Compute optimal feedforward term for unity DC gain."""
    Acl = A - B @ K
    try:
        den = float(C @ np.linalg.solve(Acl, B))[0]
        if abs(den) < 1e-10:
            return 0.0
        return -1.0 / den
    except:
        return 0.0

def _update_nbar_display(_=None):
    """Update feedforward slider when checkbox changes."""
    K = globals().get('K_current')
    A = globals().get('A_current')
    B = globals().get('B_current')
    C = globals().get('C_current')
    
    if use_feedforward.value and A is not None and B is not None and K is not None:
        nbar_opt = _compute_nbar_optimal(A, B, K, C)
        nbar_slider.value = nbar_opt
        nbar_slider.disabled = True
    else:
        nbar_slider.disabled = False
    _run_simulation()

def _run_simulation(_=None):
    """Simulate the combined system."""
    out_simulation.clear_output(wait=True)
    with out_simulation:
        try:
            A = globals().get('A_current')
            B = globals().get('B_current')
            C = globals().get('C_current')
            L = globals().get('L_current')
            K = globals().get('K_current')
            
            if A is None or B is None or C is None or L is None or K is None:
                print('Please define system, observer poles, and controller poles first.')
                return
            
            # Time vector
            t_end = 15
            dt = 0.01
            t = np.arange(0, t_end, dt)
            
            # Reference signal
            r = ref_magnitude.value * np.ones_like(t)
            nbar = nbar_slider.value if use_feedforward.value else 0.0
            
            # Initial conditions
            x = np.array([0.5, 0.2])  # True state
            x_hat = np.array([0.0, 0.0])  # Observer estimate
            
            # Simulation arrays
            x_hist = np.zeros((2, len(t)))
            x_hat_hist = np.zeros((2, len(t)))
            y_hist = np.zeros(len(t))
            y_hat_hist = np.zeros(len(t))
            u_hist = np.zeros(len(t))
            
            # Simulate
            for i, ti in enumerate(t):
                # Store current state
                x_hist[:, i] = x
                x_hat_hist[:, i] = x_hat
                
                # Output
                y = C @ x
                y_hat = C @ x_hat
                y_hist[i] = y
                y_hat_hist[i] = y_hat
                
                # Control: u = -K*x_hat + nbar*r
                u = -K @ x_hat + nbar * r[i]
                u_hist[i] = u
                
                # Update observer: x_hat_dot = A*x_hat + B*u + L*(y - y_hat)
                x_hat_dot = A @ x_hat + B.flatten() * u + L @ (y - y_hat)
                x_hat = x_hat + dt * x_hat_dot
                
                # Update true system: x_dot = A*x + B*u
                x_dot = A @ x + B.flatten() * u
                x = x + dt * x_dot
            
            # Plots
            fig, axes = plt.subplots(2, 2, figsize=(14, 10))
            
            # Observer convergence
            axes[0, 0].plot(t, x_hist[0, :], 'k-', linewidth=2, label='True $x_1$')
            axes[0, 0].plot(t, x_hat_hist[0, :], 'r--', linewidth=2, label='Estimated $\\hat{x}_1$')
            axes[0, 0].set_xlabel('Time (s)')
            axes[0, 0].set_ylabel('State')
            axes[0, 0].set_title('Observer Convergence: State $x_1$', fontweight='bold')
            axes[0, 0].legend()
            axes[0, 0].grid(True, alpha=0.3)
            
            axes[0, 1].plot(t, x_hist[1, :], 'k-', linewidth=2, label='True $x_2$')
            axes[0, 1].plot(t, x_hat_hist[1, :], 'r--', linewidth=2, label='Estimated $\\hat{x}_2$')
            axes[0, 1].set_xlabel('Time (s)')
            axes[0, 1].set_ylabel('State')
            axes[0, 1].set_title('Observer Convergence: State $x_2$', fontweight='bold')
            axes[0, 1].legend()
            axes[0, 1].grid(True, alpha=0.3)
            
            # Output tracking
            axes[1, 0].plot(t, y_hist, 'b-', linewidth=2, label='Output $y(t)$')
            axes[1, 0].plot(t, r, 'g--', linewidth=2, label=f'Reference $r(t) = {ref_magnitude.value:.2f}$')
            axes[1, 0].set_xlabel('Time (s)')
            axes[1, 0].set_ylabel('Output')
            axes[1, 0].set_title('Output Tracking', fontweight='bold')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
            
            # Control input
            axes[1, 1].plot(t, u_hist, 'purple', linewidth=2, label='Control Input $u(t)$')
            axes[1, 1].set_xlabel('Time (s)')
            axes[1, 1].set_ylabel('Input')
            axes[1, 1].set_title('Control Input', fontweight='bold')
            axes[1, 1].legend()
            axes[1, 1].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Print summary
            print(f'\nSimulation Summary:')
            print(f'Reference: {ref_magnitude.value:.3f}')
            print(f'Final output: {y_hist[-1]:.3f}')
            print(f'Steady-state error: {(ref_magnitude.value - y_hist[-1]):.3f}')
            print(f'Max control input: {np.max(np.abs(u_hist)):.3f}')
            if use_feedforward.value:
                print(f'Feedforward term Nbar = {nbar:.3f}')
            
        except Exception as e:
            print(f'Simulation error: {e}')
            import traceback
            traceback.print_exc()

use_feedforward.observe(_update_nbar_display, names='value')
nbar_slider.observe(_run_simulation, names='value')
ref_magnitude.observe(_run_simulation, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Simulation Controls</b>'),
    ref_magnitude,
    widgets.HBox([use_feedforward]),
    widgets.VBox([widgets.HTML('Feedforward Gain:'), nbar_slider]),
    out_simulation
]))

_run_simulation()